###  CASE 11 — Rect 4×8 + 3×3 — Convolution TopLeft:

In [1]:
%%writefile case11_rect_3x3_convTopLeft.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define WIDTH       8
#define HEIGHT      4
#define MASK_WIDTH  3
#define MASK_HEIGHT 3
#define BLOCK_X     8
#define BLOCK_Y     4

__global__ void convolutionTopLeft(int *dA, int *dMask, int *dC,
                                   int width, int height,
                                   int mWidth, int mHeight)
{
    int col = threadIdx.x + blockIdx.x * blockDim.x;
    int row = threadIdx.y + blockIdx.y * blockDim.y;
    if (row < height && col < width)
    {
        int sum = 0;
        for (int i = 0; i < mHeight; i++)
            for (int j = 0; j < mWidth; j++)
            {
                int r = row + i;
                int c = col + j;
                if (r < height && c < width)
                    sum += dA[r*width+c] *
                           dMask[(mHeight-1-i)*mWidth+(mWidth-1-j)];
            }
        dC[row*width+col] = sum;
    }
}

void printMatrix(const char *label, int *M, int w, int h)
{
    printf("\n%s:\n", label);
    for (int r = 0; r < h; r++)
    {
        for (int c = 0; c < w; c++)
            printf("%6d", M[r*w+c]);
        printf("\n");
    }
}

int main()
{
    int size     = WIDTH * HEIGHT * sizeof(int);
    int maskSize = MASK_WIDTH * MASK_HEIGHT * sizeof(int);

    int *hA    = (int*) malloc(size);
    int *hMask = (int*) malloc(maskSize);
    int *hC    = (int*) malloc(size);

    srand(time(NULL));
    for (int i = 0; i < WIDTH*HEIGHT; i++)
        hA[i] = rand()%9+1;

    int tempMask[3][3] = {{1,0,-1},{2,0,-2},{1,0,-1}};
    for (int i = 0; i < MASK_HEIGHT; i++)
        for (int j = 0; j < MASK_WIDTH; j++)
            hMask[i*MASK_WIDTH+j] = tempMask[i][j];

    printMatrix("Input Matrix (4 rows x 8 cols)", hA,    WIDTH,      HEIGHT);
    printMatrix("Mask (3x3)",                     hMask, MASK_WIDTH, MASK_HEIGHT);

    int *dA, *dMask, *dC;
    cudaMalloc((void**)&dA,    size);
    cudaMalloc((void**)&dMask, maskSize);
    cudaMalloc((void**)&dC,    size);
    cudaMemcpy(dA,    hA,    size,     cudaMemcpyHostToDevice);
    cudaMemcpy(dMask, hMask, maskSize, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_X, BLOCK_Y, 1);
    dim3 DimGrid((int)ceil((float)WIDTH/BLOCK_X),
                 (int)ceil((float)HEIGHT/BLOCK_Y), 1);

    cudaEvent_t start, stop; float gpuTime;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);

    convolutionTopLeft<<<DimGrid,DimBlock>>>(dA,dMask,dC,
                        WIDTH,HEIGHT,MASK_WIDTH,MASK_HEIGHT);

    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpuTime, start, stop);
    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    printMatrix("OUTPUT: Convolution TopLeft (4x8 rect, 3x3)", hC, WIDTH, HEIGHT);
    printf("\nGPU Time: %.4f ms\n", gpuTime);
    printf("Grid: %dx%d  Block: %dx%d\n",
            DimGrid.x,DimGrid.y,DimBlock.x,DimBlock.y);

    cudaFree(dA); cudaFree(dMask); cudaFree(dC);
    free(hA); free(hMask); free(hC);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return 0;
}

Writing case11_rect_3x3_convTopLeft.cu


### Compile and Run:

In [2]:
!nvcc -arch=sm_75 case11_rect_3x3_convTopLeft.cu -o case11_rect_3x3_convTopLeft

!./case11_rect_3x3_convTopLeft


Input Matrix (4 rows x 8 cols):
     9     4     5     4     9     2     5     4
     5     5     6     6     5     8     7     7
     4     5     5     6     8     8     6     7
     3     1     9     1     6     8     3     4

Mask (3x3):
     1     0    -1
     2     0    -2
     1     0    -1

OUTPUT: Convolution TopLeft (4x8 rect, 3x3):
    -1     3     5     4    -2    -1   -25   -25
     9     3     2    13    -5    -7   -22   -25
    13     1    -3    16    -8    -9   -12   -15
     6     0    -3     7    -3    -4    -3    -4

GPU Time: 0.1083 ms
Grid: 1x1  Block: 8x4
